# USLegalQA — Baselines B1 (closed-book) and B2 (open-book)

**v2 of this notebook.** Changes from v1:
- Generation backs off automatically on out-of-memory instead of aborting
- Per-system batch sizes: B2 prompts carry ~1,800 characters of passage, so
  their attention memory is several times B1's and a batch size that suits B1
  exhausts a T4 partway through B2
- GPU memory is released between systems
- `category` is carried through to the output, so per-category evaluation works
  without a separate repair step

---

**B1 — closed-book.** Question only; the opinion is **not** supplied. Anything
answered correctly comes from pre-training memory.

**B2 — open-book.** Question plus the source passage. Measures reading
comprehension.

**The gap between them is the contamination diagnostic.**

---

### Before running
1. Accelerator → **GPU T4 x2**
2. Internet → **On**
3. Add Data → your dataset containing `test_with_passages.jsonl`
4. Add-ons → Secrets → `HF_TOKEN` (ticked) for gated models

In [1]:
# --- configuration ---

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"   # gated: needs HF_TOKEN
# Ungated alternative:
# MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# If B1 is already generated and downloaded, leave it out to save ~30 minutes.
SYSTEMS = ["b2"]              # set to ["b1", "b2"] to generate both

# B2 prompts are several times larger than B1's, so they need a smaller batch.
BATCH_SIZE = {"b1": 16, "b2": 4}

LIMIT          = None         # e.g. 200 for a trial; None = full split
MAX_NEW_TOKENS = 160
LOAD_IN_4BIT   = True

INPUT_DIR  = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

print(f"model    {MODEL_ID}")
print(f"systems  {SYSTEMS}")
print(f"batch    {BATCH_SIZE}")
print(f"limit    {LIMIT or 'full split'}")

model    meta-llama/Llama-3.2-3B-Instruct
systems  ['b2']
batch    {'b1': 16, 'b2': 4}
limit    full split


## 1. Environment

In [2]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "accelerate", "bitsandbytes"], check=False)

import torch, transformers
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("cuda        ", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB")
else:
    print("\nNO GPU -- set Settings -> Accelerator -> GPU T4 x2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.1 MB/s eta 0:00:00
torch        2.10.0+cu128
transformers 5.17.0
cuda         True
  GPU 0: Tesla T4, 15.6 GB
  GPU 1: Tesla T4, 15.6 GB


## 2. Load the test data

In [3]:
import json, os, glob

def find_input(filename):
    hits = glob.glob(f"{INPUT_DIR}/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(
            f"{filename} not found under {INPUT_DIR}.\n"
            "Attach the dataset via Add Data in the right-hand panel.")
    return hits[0]

path = find_input("test_with_passages.jsonl")
print(f"reading {path}")

items = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
if LIMIT:
    items = items[:LIMIT]

print(f"{len(items)} items across {len({i['cluster_id'] for i in items})} opinions")

missing = {"question", "answer", "passage", "case_name"} - set(items[0])
assert not missing, f"records are missing {missing}"

# Category is needed for per-category evaluation. If it is absent here, run
# `python -m uslegalqa.prepare_kaggle` again after backfilling the corpus.
n_cat = sum(1 for i in items if i.get("category"))
print(f"category present on {n_cat}/{len(items)} items")
if n_cat < len(items):
    print("  WARNING: some items lack a category; per-category tables will be "
          "incomplete unless patched locally afterwards")

print(f"\nExample: {items[0]['case_name']}")
print(f"  Q: {items[0]['question']}")
print(f"  passage: {len(items[0]['passage'])} chars")

reading /kaggle/input/datasets/minhalzafar/uslegalqa-test/test_with_passages.jsonl
1771 items across 109 opinions
category present on 611/1771 items

Example: Wilkinson v. Dotson
  Q: What constitutional violations did Dotson claim resulted from Ohio's parole procedures?
  passage: 1972 chars


## 3. Prompts

Identical to `baselines.py`. B1 and B2 differ **only** in whether the passage is
supplied — any other difference would confound the diagnostic.

In [4]:
CLOSED_BOOK = """You are a US legal expert. Answer the question about the Supreme Court case below.

CASE: {case_name} ({date_filed})
QUESTION: {question}

Answer in 20-100 words. State what the Court held or reasoned. Do not speculate; if you do not know, say so."""

OPEN_BOOK = """You are a US legal expert. Answer the question using ONLY the passage below.

CASE: {case_name} ({date_filed})

PASSAGE:
\"\"\"
{passage}
\"\"\"

QUESTION: {question}

Answer in 20-100 words, based only on the passage. State the point in your own words rather than quoting at length."""


def build_prompt(system, item):
    if system == "b2":
        return OPEN_BOOK.format(case_name=item["case_name"],
                                date_filed=item.get("date_filed", ""),
                                passage=item["passage"],
                                question=item["question"])
    return CLOSED_BOOK.format(case_name=item["case_name"],
                              date_filed=item.get("date_filed", ""),
                              question=item["question"])


# The diagnostic is meaningless if B1 leaks the passage. Fail loudly here
# rather than produce a confidently wrong answer.
snippet = items[0]["passage"][:80]
assert snippet not in build_prompt("b1", items[0]), "B1 LEAKS THE PASSAGE"
assert snippet in build_prompt("b2", items[0]), "B2 is missing the passage"
print("B1 withholds the passage, B2 supplies it.")

for s in SYSTEMS:
    lens = [len(build_prompt(s, i)) for i in items]
    print(f"  {s}: prompt chars min {min(lens):,} / "
          f"median {sorted(lens)[len(lens)//2]:,} / max {max(lens):,}")

B1 withholds the passage, B2 supplies it.
  b2: prompt chars min 1,381 / median 2,553 / max 7,672


## 4. Load the model

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception:
    print("No HF_TOKEN secret -- fine for ungated models")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)

# Left padding is required for batched generation with a decoder-only model:
# with right padding, generation continues from pad tokens and the output is
# garbage for every sequence shorter than the longest in the batch.
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

kwargs = {"device_map": "auto", "token": hf_token}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
else:
    kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
model.eval()
print(f"\nloaded {MODEL_ID}")

HF_TOKEN loaded from Kaggle Secrets


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]


loaded meta-llama/Llama-3.2-3B-Instruct


## 5. Generate

Batched greedy decoding with automatic back-off on out-of-memory. Checkpointed
every 10 batches and resumable, so an interrupted session costs only the
current batch.

In [6]:
import time, gc


def _generate(prompts):
    texts = [
        tokenizer.apply_chat_template([{"role": "user", "content": p}],
                                      tokenize=False, add_generation_prompt=True)
        for p in prompts
    ]
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=4096).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False,                  # greedy
                             pad_token_id=tokenizer.pad_token_id)
    # Slice off the prompt: scoring the full sequence would inflate overlap.
    gen = out[:, enc["input_ids"].shape[-1]:]
    texts_out = [t.strip() for t in
                 tokenizer.batch_decode(gen, skip_special_tokens=True)]
    del enc, out, gen
    return texts_out


def generate_batch(prompts, size):
    """Generate, halving the batch and retrying whenever memory runs out.

    Passage length varies across the corpus, so a batch size that works for
    most of the split can still exhaust the GPU on a run of long passages.
    Backing off is preferable to aborting a run that is most of the way done.
    """
    while size >= 1:
        try:
            out = []
            for i in range(0, len(prompts), size):
                out += _generate(prompts[i:i+size])
            return out
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            size //= 2
            print(f"    out of memory -- retrying at batch size {size}",
                  flush=True)
    raise RuntimeError("out of memory even at batch size 1")


def run_system(system):
    gc.collect(); torch.cuda.empty_cache()
    out_path = f"{OUTPUT_DIR}/{system}.jsonl"
    size = BATCH_SIZE[system] if isinstance(BATCH_SIZE, dict) else BATCH_SIZE

    done = set()
    if os.path.exists(out_path):
        for line in open(out_path, encoding="utf-8"):
            if line.strip():
                try:
                    d = json.loads(line)
                    done.add((d["cluster_id"], d["question"]))
                except Exception:
                    pass
        if done:
            print(f"[{system}] resuming: {len(done)} already written")

    todo = [i for i in items if (i["cluster_id"], i["question"]) not in done]
    if not todo:
        print(f"[{system}] complete")
        return

    print(f"[{system}] {len(todo)} items, batch size {size}")
    start, empty = time.time(), 0

    with open(out_path, "a", encoding="utf-8") as f:
        for b in range(0, len(todo), size):
            batch = todo[b:b+size]
            preds = generate_batch([build_prompt(system, i) for i in batch], size)

            for item, pred in zip(batch, preds):
                if not pred.strip():
                    empty += 1
                f.write(json.dumps({
                    "cluster_id": item["cluster_id"],
                    "question": item["question"],
                    "answer": item["answer"],
                    "prediction": pred,
                    "question_type": item.get("question_type"),
                    "category": item.get("category"),
                    "system": system,
                }, ensure_ascii=False) + "\n")

            if (b // size) % 10 == 0:
                f.flush()
                n = b + len(batch)
                rate = n / max(time.time() - start, 1e-9)
                print(f"  [{n}/{len(todo)}] {rate:.1f} items/s, "
                      f"~{(len(todo)-n)/max(rate,1e-9)/60:.0f} min left, "
                      f"{empty} empty", flush=True)

    print(f"[{system}] done in {(time.time()-start)/60:.1f} min, "
          f"{empty} empty -> {out_path}")


for system in SYSTEMS:
    run_system(system)

[b2] 1771 items, batch size 4


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  [4/1771] 0.3 items/s, ~101 min left, 0 empty
  [44/1771] 0.3 items/s, ~104 min left, 0 empty
  [84/1771] 0.3 items/s, ~104 min left, 0 empty
  [124/1771] 0.3 items/s, ~103 min left, 0 empty
  [164/1771] 0.3 items/s, ~101 min left, 0 empty
  [204/1771] 0.3 items/s, ~100 min left, 0 empty
  [244/1771] 0.3 items/s, ~97 min left, 0 empty
  [284/1771] 0.3 items/s, ~95 min left, 0 empty
  [324/1771] 0.3 items/s, ~92 min left, 0 empty
  [364/1771] 0.3 items/s, ~90 min left, 0 empty
  [404/1771] 0.3 items/s, ~87 min left, 0 empty
  [444/1771] 0.3 items/s, ~85 min left, 0 empty
  [484/1771] 0.3 items/s, ~82 min left, 0 empty
  [524/1771] 0.3 items/s, ~79 min left, 0 empty
  [564/1771] 0.3 items/s, ~77 min left, 0 empty
  [604/1771] 0.3 items/s, ~75 min left, 0 empty
  [644/1771] 0.3 items/s, ~72 min left, 0 empty
  [684/1771] 0.3 items/s, ~70 min left, 0 empty
  [724/1771] 0.3 items/s, ~67 min left, 0 empty
  [764/1771] 0.3 items/s, ~64 min left, 0 empty
  [804/1771] 0.3 items/s, ~62 min left

## 6. Inspect before downloading

In [7]:
for system in SYSTEMS:
    path = f"{OUTPUT_DIR}/{system}.jsonl"
    if not os.path.exists(path):
        continue

    recs = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    lens = [len(r["prediction"].split()) for r in recs]
    ref_lens = [len(r["answer"].split()) for r in recs]
    empty = sum(1 for l in lens if l == 0)
    dupes = len(recs) - len({r["prediction"] for r in recs})

    print(f"\n{'='*66}")
    print(f"{system}: {len(recs)} predictions")
    print(f"{'='*66}")
    print(f"  mean length  {sum(lens)/len(lens):.1f} words "
          f"(reference {sum(ref_lens)/len(ref_lens):.1f})")
    print(f"  empty        {empty}")
    print(f"  duplicates   {dupes} ({100*dupes/len(recs):.1f}%)")
    print(f"  category set {sum(1 for r in recs if r.get('category'))}/{len(recs)}")

    ratio = (sum(lens)/len(lens)) / max(sum(ref_lens)/len(ref_lens), 1e-9)
    if ratio > 2:
        print(f"  WARNING: {ratio:.1f}x reference length -- check the EOS token")
    if empty > len(recs) * 0.05:
        print(f"  WARNING: {100*empty/len(recs):.1f}% empty")

    r = recs[0]
    print(f"\n  Q:          {r['question'][:100]}")
    print(f"  reference:  {r['answer'][:150]}")
    print(f"  prediction: {r['prediction'][:150]}")


b2: 1771 predictions
  mean length  91.8 words (reference 58.6)
  empty        0
  duplicates   0 (0.0%)
  category set 611/1771

  Q:          What constitutional violations did Dotson claim resulted from Ohio's parole procedures?
  reference:  Dotson contended that applying parole guidelines adopted in 1998 retroactively to his case, which predated those guidelines, violated two constitution
  prediction: Dotson claimed that Ohio's parole procedures violated the Constitution's Ex Post Facto Clause and Due Process Clause. Specifically, he alleged that th


## 7. Download and score

Files appear under **Output** in the right-hand panel. Put them in
`data/predictions/` locally, then:

```powershell
python -m uslegalqa.patch_predictions          # only if category was missing
python -m uslegalqa.evaluate `
  --preds data/predictions/b0.jsonl `
  --preds data/predictions/b1.jsonl `
  --preds data/predictions/b2.jsonl `
  --reference b1 --breakdowns
```

### Reading the result

Reference points already measured: floor −0.012, B0 extractive −0.041,
B1 closed-book +0.082 (rescaled BERTScore).

**A large, significant B1→B2 gap** means supplying the opinion is what produces
correct answers: the dataset tests comprehension and the benchmark is sound.

**A small gap** means the model is not making much use of the passage, and the
contamination caveat carries into the discussion.

Also worth checking: B1 scored 0.2435 on criminal procedure against 0.1773 on
judicial power. If B2 flattens that spread, the gradient in B1 was pre-training
exposure rather than task difficulty.

In [8]:
print("Ready for download:\n")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".jsonl"):
        n = sum(1 for _ in open(f"{OUTPUT_DIR}/{f}", encoding="utf-8"))
        mb = os.path.getsize(f"{OUTPUT_DIR}/{f}") / 1e6
        print(f"  {f:<16} {n:>6} records  {mb:>6.2f} MB")

Ready for download:

  b2.jsonl           1771 records    2.26 MB
